# Notebook 03: Vocabulary + Pairs (XuetangX)

**Purpose:** Build course vocabulary + (prefix → next item) pairs for next-item prediction.

**Inputs:** `data/interim/xuetangx.duckdb` → view `xuetangx_events_sessionized`

**Outputs:**
- `data/processed/xuetangx/vocab/course2id.json`
- `data/processed/xuetangx/vocab/id2course.json`
- `data/processed/xuetangx/pairs/pairs.parquet`
- DuckDB view: `xuetangx_pairs`

In [1]:
# [CELL 03-00] Bootstrap

import json, time, uuid, hashlib
from pathlib import Path
from datetime import datetime
from typing import Any, Dict, List
import numpy as np
import pandas as pd

def find_repo_root(start):
    for p in [Path(start).resolve(), *Path(start).resolve().parents]:
        if (p / 'PROJECT_STATE.md').exists(): return p
    raise RuntimeError('PROJECT_STATE.md not found')

REPO_ROOT = find_repo_root(Path.cwd())
PATHS = {
    'META_REGISTRY':  REPO_ROOT / 'meta.json',
    'DATA_INTERIM':   REPO_ROOT / 'data' / 'interim',
    'DATA_PROCESSED': REPO_ROOT / 'data' / 'processed',
    'REPORTS':        REPO_ROOT / 'reports',
}

def cell_start(cid, title, **kw):
    t = time.time()
    print(f'\n[{cid}] {title}')
    print(f'[{cid}] start={datetime.now().isoformat(timespec="seconds")}')
    for k,v in kw.items(): print(f'[{cid}] {k}={v}')
    return t

def cell_end(cid, t0, **kw):
    for k,v in kw.items(): print(f'[{cid}] {k}={v}')
    print(f'[{cid}] elapsed={time.time()-t0:.2f}s  done')

def write_json_atomic(path, obj, indent=2):
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + f'.tmp_{uuid.uuid4().hex}')
    tmp.write_text(json.dumps(obj, ensure_ascii=False, indent=indent), encoding='utf-8')
    tmp.replace(path)

def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))

def sha256_file(path):
    h = hashlib.sha256()
    with open(path,'rb') as f:
        while chunk := f.read(1<<20): h.update(chunk)
    return h.hexdigest()

print(f'[CELL 03-00] REPO_ROOT={REPO_ROOT}  done')

[CELL 03-00] REPO_ROOT=/home/user/jamalla/anonymous-users-mooc-session-meta  done


In [2]:
# [CELL 03-01] Seed + run config

import random
GLOBAL_SEED = 20260107
random.seed(GLOBAL_SEED); np.random.seed(GLOBAL_SEED)

NOTEBOOK_NAME = '03_vocab_pairs_xuetangx'
DATASET       = 'xuetangx'
RUN_TAG = datetime.now().strftime('%Y%m%d_%H%M%S')
RUN_ID  = uuid.uuid4().hex

OUT_DIR       = PATHS['REPORTS'] / NOTEBOOK_NAME / RUN_TAG
OUT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_PATH   = OUT_DIR / 'report.json'
CONFIG_PATH   = OUT_DIR / 'config.json'
MANIFEST_PATH = OUT_DIR / 'manifest.json'

DUCKDB_PATH = PATHS['DATA_INTERIM'] / 'xuetangx.duckdb'
EVENTS_VIEW = 'xuetangx_events_sessionized'

VOCAB_DIR = PATHS['DATA_PROCESSED'] / 'xuetangx' / 'vocab'
PAIRS_DIR = PATHS['DATA_PROCESSED'] / 'xuetangx' / 'pairs'
VOCAB_DIR.mkdir(parents=True, exist_ok=True)
PAIRS_DIR.mkdir(parents=True, exist_ok=True)

OUT_COURSE2ID = VOCAB_DIR / 'course2id.json'
OUT_ID2COURSE = VOCAB_DIR / 'id2course.json'
OUT_PAIRS     = PAIRS_DIR / 'pairs.parquet'

CFG = {'notebook': NOTEBOOK_NAME, 'dataset': DATASET, 'seed': GLOBAL_SEED,
       'pairs': {'min_prefix_len': 1, 'deduplicate_consecutive': True}}
write_json_atomic(CONFIG_PATH, CFG)

for path, obj in [(REPORT_PATH, {'run_id':RUN_ID,'notebook':NOTEBOOK_NAME,'run_tag':RUN_TAG,
                                   'created_at':datetime.now().isoformat(timespec='seconds'),
                                   'metrics':{},'key_findings':[],'sanity_samples':{},'data_fingerprints':{},'notes':[]}),
                   (MANIFEST_PATH, {'run_id':RUN_ID,'notebook':NOTEBOOK_NAME,'run_tag':RUN_TAG,'artifacts':[]})]: 
    write_json_atomic(path, obj)

META = PATHS['META_REGISTRY']
if not META.exists(): write_json_atomic(META, {'schema_version':1,'runs':[]})
m = read_json(META); m['runs'].append({'run_id':RUN_ID,'notebook':NOTEBOOK_NAME,'run_tag':RUN_TAG,'out_dir':str(OUT_DIR)})
write_json_atomic(META, m)
print(f'[CELL 03-01] run={RUN_TAG}  done')

[CELL 03-01] run=20260408_130051  done


In [3]:
# [CELL 03-02] Load sessionized events

import duckdb
t0 = cell_start('CELL 03-02', 'Load events from DuckDB')

if not DUCKDB_PATH.exists():
    raise RuntimeError(f'Missing {DUCKDB_PATH}. Run 02_sessionize_xuetangx.ipynb first.')

con = duckdb.connect(str(DUCKDB_PATH), read_only=True)
events = con.execute(f"""
    SELECT user_id, course_id, session_id, ts_epoch, pos_in_sess
    FROM {EVENTS_VIEW}
    ORDER BY user_id, session_id, ts_epoch, pos_in_sess
""").fetchdf()
con.close()

print(f'[CELL 03-02] events shape: {events.shape}')
cell_end('CELL 03-02', t0, n_events=len(events))


[CELL 03-02] Load events from DuckDB
[CELL 03-02] start=2026-04-08T13:00:52


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[CELL 03-02] events shape: (27875155, 5)
[CELL 03-02] n_events=27875155
[CELL 03-02] elapsed=15.48s  done


In [4]:
# [CELL 03-03] Build course vocabulary (alphabetical → deterministic)

t0 = cell_start('CELL 03-03', 'Build vocabulary')

unique_courses = sorted(events['course_id'].unique())
course2id = {c: i for i, c in enumerate(unique_courses)}
id2course  = {i: c for c, i in course2id.items()}
n_items = len(course2id)

write_json_atomic(OUT_COURSE2ID, course2id)
write_json_atomic(OUT_ID2COURSE, id2course)

print(f'[CELL 03-03] n_items={n_items}')
print(f'[CELL 03-03] First 3: {unique_courses[:3]}')
cell_end('CELL 03-03', t0, n_items=n_items)


[CELL 03-03] Build vocabulary
[CELL 03-03] start=2026-04-08T13:01:07
[CELL 03-03] n_items=1517
[CELL 03-03] First 3: ['AdelaideX/humbio101x/_', 'BIT/ELC05198/2014_T2', 'BIT/PHY1701701/2015_T1']
[CELL 03-03] n_items=1517
[CELL 03-03] elapsed=1.96s  done


In [5]:
# [CELL 03-04] Map course → item_id; build session sequences (dedup consecutive)

t0 = cell_start('CELL 03-04', 'Map IDs + build session sequences')

events['item_id'] = events['course_id'].map(course2id).astype(int)
assert events['item_id'].isna().sum() == 0, 'Unmapped courses found'

DEDUPE = CFG['pairs']['deduplicate_consecutive']

def dedupe_consecutive(items, tss):
    di, dt = [items[0]], [tss[0]]
    for i in range(1, len(items)):
        if items[i] != di[-1]: di.append(items[i]); dt.append(tss[i])
    return di, dt

session_seqs = []
for (uid, sid), g in events.groupby(['user_id','session_id']):
    g = g.sort_values('ts_epoch')
    items = g['item_id'].tolist()
    tss   = g['ts_epoch'].tolist()
    if DEDUPE: items, tss = dedupe_consecutive(items, tss)
    session_seqs.append({'user_id':uid,'session_id':sid,'item_seq':items,'ts_seq':tss})

print(f'[CELL 03-04] {len(session_seqs):,} session sequences')
cell_end('CELL 03-04', t0)


[CELL 03-04] Map IDs + build session sequences
[CELL 03-04] start=2026-04-08T13:01:09
[CELL 03-04] 906,341 session sequences
[CELL 03-04] elapsed=181.47s  done


In [6]:
# [CELL 03-05] Create prefix→label pairs (chronological, no leakage)

t0 = cell_start('CELL 03-05', 'Create prefix→label pairs')

MIN_PFX = CFG['pairs']['min_prefix_len']
pairs = []
pid = 0

for sess in session_seqs:
    items = sess['item_seq']
    tss   = sess['ts_seq']
    if len(items) < MIN_PFX + 1: continue
    for t in range(MIN_PFX, len(items)):
        pfx_ts = tss[:t]
        lbl_ts = tss[t]
        if max(pfx_ts) >= lbl_ts: continue  # strict chronological
        pairs.append({'pair_id': pid, 'user_id': sess['user_id'], 'session_id': sess['session_id'],
                       'prefix': items[:t], 'label': int(items[t]),
                       'label_ts_epoch': int(lbl_ts), 'prefix_len': t})
        pid += 1

pairs_df = pd.DataFrame(pairs)
print(f'[CELL 03-05] {len(pairs_df):,} pairs created')
print(f'[CELL 03-05] prefix_len p50={pairs_df["prefix_len"].quantile(.5):.0f}  p90={pairs_df["prefix_len"].quantile(.9):.0f}  max={pairs_df["prefix_len"].max()}')
cell_end('CELL 03-05', t0, n_pairs=len(pairs_df))


[CELL 03-05] Create prefix→label pairs
[CELL 03-05] start=2026-04-08T13:04:10
[CELL 03-05] 487,696 pairs created
[CELL 03-05] prefix_len p50=2  p90=14  max=1000
[CELL 03-05] n_pairs=487696
[CELL 03-05] elapsed=2.35s  done


In [7]:
# [CELL 03-06] Validate pairs

t0 = cell_start('CELL 03-06', 'Validation checks')

assert (pairs_df['label'] < n_items).all(), 'Label out of vocab'
all_pfx = [x for pfx in pairs_df['prefix'] for x in pfx]
assert max(all_pfx) < n_items, 'Prefix item out of vocab'
assert (pairs_df['prefix_len'] >= MIN_PFX).all(), 'Short prefix found'
print(f'[CELL 03-06] All labels and prefix items in vocab [0,{n_items-1}]')

user_counts = pairs_df.groupby('user_id').size()
print(f'[CELL 03-06] Users with pairs: {len(user_counts):,}  min={user_counts.min()}  p50={user_counts.quantile(.5):.0f}  max={user_counts.max()}')
cell_end('CELL 03-06', t0)


[CELL 03-06] Validation checks
[CELL 03-06] start=2026-04-08T13:04:13
[CELL 03-06] All labels and prefix items in vocab [0,1516]
[CELL 03-06] Users with pairs: 70,809  min=1  p50=3  max=4666
[CELL 03-06] elapsed=0.26s  done


In [9]:
# [CELL 03-07] Save pairs.parquet + register DuckDB view

t0 = cell_start('CELL 03-07', 'Save + register DuckDB view')

pairs_df.to_parquet(OUT_PAIRS, index=False, compression='zstd')
pairs_bytes = int(OUT_PAIRS.stat().st_size)
pairs_sha   = sha256_file(OUT_PAIRS)
#print(f'[CELL 03-07] Saved {OUT_PAIRS}  ({pairs_bytes/1<<20:.1f} MB)')
print(f'[CELL 03-07] Saved {OUT_PAIRS}  ({pairs_bytes / (1 << 20):.1f} MB)')

def esc(p): return str(p).replace("'","''")
con = duckdb.connect(str(DUCKDB_PATH), read_only=False)
con.execute('DROP VIEW IF EXISTS xuetangx_pairs;')
con.execute(f"CREATE VIEW xuetangx_pairs AS SELECT * FROM read_parquet('{esc(OUT_PAIRS)}')")
n = con.execute('SELECT COUNT(*) FROM xuetangx_pairs').fetchone()[0]
con.close()
print(f'[CELL 03-07] View xuetangx_pairs: {n:,} rows')

# Update report
r = read_json(REPORT_PATH)
r['metrics'] = {'n_items': n_items, 'n_pairs': len(pairs_df), 'n_users': int(len(user_counts))}
r['data_fingerprints']['pairs'] = {'path': str(OUT_PAIRS), 'bytes': pairs_bytes, 'sha256': pairs_sha}
r['sanity_samples']['pairs_head3'] = pairs_df.head(3).to_dict(orient='records')
write_json_atomic(REPORT_PATH, r)

cell_end('CELL 03-07', t0, n_pairs=len(pairs_df))


[CELL 03-07] Save + register DuckDB view
[CELL 03-07] start=2026-04-08T13:51:18
[CELL 03-07] Saved /home/user/jamalla/anonymous-users-mooc-session-meta/data/processed/xuetangx/pairs/pairs.parquet  (6.5 MB)
[CELL 03-07] View xuetangx_pairs: 487,696 rows
[CELL 03-07] n_pairs=487696
[CELL 03-07] elapsed=1.16s  done


## Notebook 03 Complete
**Next:** `03b_srs_scores_xuetangx.ipynb` — compute SRS scores and attach to pairs.